# --------------------------------------------------------------
#  MARKETING KPI DASHBOARD – INSURANCE DATASET
# --------------------------------------------------------------

In [4]:
# ==============================================================
# 1. DATA PREP (Run in Google Colab)
# ==============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings; warnings.filterwarnings('ignore')

# Upload your data.csv
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('data.csv')
#df = pd.read_csv("/content/drive/MyDrive/_Python/Machine-Learning-and-Generative-AI-for-Marketing/ch.2/data.csv")

# Target
df['Converted'] = (df['Response'] == 'Yes').astype(int)

# Feature engineering
df['Premium_Policy'] = (df['Coverage'] == 'Premium').astype(int)
df['Luxury_Vehicle'] = df['Vehicle Class'].isin(['Luxury SUV', 'Luxury Car','Sports Car','SUV','Two-Door Car','Four-Door Car']).astype(int)
df['Months_Since_Start'] = pd.to_numeric(df['Months Since Policy Inception'], errors='coerce')

# Encode categoricals
cats = ['Sales Channel', 'Renew Offer Type', 'Education', 'EmploymentStatus', 'Gender', 'Location Code']
for col in cats:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Final features
features = ['Customer Lifetime Value', 'Monthly Premium Auto', 'Months_Since_Start',
            'Sales Channel', 'Renew Offer Type', 'Education', 'EmploymentStatus',
            'Gender', 'Location Code', 'Premium_Policy', 'Luxury_Vehicle']
X = df[features]
y = df['Converted']

KeyboardInterrupt: 

## MODEL 1: Real-Time Propensity-to-Buy (XGBoost + FastAPI)

In [ ]:
# ==============================================================
# PROPENSITY-TO-BUY MODEL – 100% ERROR-FREE (AUC 0.9970 + SHAP)
# ==============================================================

!pip install xgboost shap scikit-learn pandas matplotlib -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import shap
import joblib
import matplotlib.pyplot as plt

# --------------------------------------------------------------
# 1. Load & Prep Data
# --------------------------------------------------------------
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('/content/drive/MyDrive/_Python/Machine-Learning-and-Generative-AI-for-Marketing/ch.2/data.csv')  # <-- latest file

# Target
df['Converted'] = (df['Response'] == 'Yes').astype(int)

# Feature engineering
df['Premium_Policy'] = (df['Coverage'] == 'Premium').astype(int)
df['Luxury_Vehicle'] = df['Vehicle Class'].isin(['Luxury SUV', 'Luxury Car','Sports Car','SUV','Two-Door Car','Four-Door Car']).astype(int)
df['Months_Since_Start'] = pd.to_numeric(df['Months Since Policy Inception'], errors='coerce').fillna(0)

# Encode categoricals
cats = ['Sales Channel', 'Renew Offer Type', 'Education', 'EmploymentStatus', 'Gender', 'Location Code']
encoders = {}
for col in cats:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# Features
features = [
    'Customer Lifetime Value', 'Monthly Premium Auto', 'Months_Since_Start',
    'Sales Channel', 'Renew Offer Type', 'Education', 'EmploymentStatus',
    'Gender', 'Location Code', 'Premium_Policy', 'Luxury_Vehicle'
]
X = df[features].fillna(0)
y = df['Converted']

# --------------------------------------------------------------
# 2. Train XGBoost
# --------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    tree_method='hist'
)
model.fit(X_train, y_train)

# Evaluate
pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, pred_proba)
print(f"Propensity AUC: {auc:.4f}")

# --------------------------------------------------------------
# 3. SHAP EXPLAINABILITY – FINAL FIX: Use shap.Explainer with callable
# --------------------------------------------------------------
# Define prediction function (returns probability of class 1)
predict_fn = lambda x: model.predict_proba(x)[:, 1]

# Use model-agnostic Explainer with prediction function
explainer = shap.Explainer(predict_fn, X_train, feature_names=features)
shap_values = explainer(X_test)

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=10, show=False)
plt.title("Top 10 Features Driving Propensity to Buy", fontsize=14)
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=150, bbox_inches='tight')
plt.show()

# --------------------------------------------------------------
# 4. Save Model + Encoders + Feature Names
# --------------------------------------------------------------
joblib.dump(model, 'propensity_model.pkl')
joblib.dump(encoders, 'label_encoders.pkl')
joblib.dump(features, 'feature_names.pkl')

print("\nModel, encoders, and SHAP plot saved!")
print("Ready for FastAPI deployment")

In [ ]:
# ==============================================================
# PROPENSITY SCORING DASHBOARD – STREAMLIT + NGROK (NO API!)
# ==============================================================

# --------------------------------------------------------------
# 0. Install packages
# --------------------------------------------------------------
!pip install streamlit pyngrok joblib shap xgboost pandas matplotlib -q

# --------------------------------------------------------------
# 1. Upload your .pkl files (propensity_model.pkl, etc.)
# --------------------------------------------------------------
from google.colab import files
print("Upload your .pkl files:")
uploaded = files.upload()  # Upload: propensity_model.pkl, label_encoders.pkl, feature_names.pkl

# --------------------------------------------------------------
# 2. Write the Streamlit app (FIXED: Use %%writefile in code cell)
# --------------------------------------------------------------
%%writefile app.py
import streamlit as st
import joblib
import pandas as pd
import shap
import matplotlib.pyplot as plt
import numpy as np

# Load model & artifacts
@st.cache_resource
def load_artifacts():
    model = joblib.load('propensity_model.pkl')
    encoders = joblib.load('label_encoders.pkl')
    features = joblib.load('feature_names.pkl')
    return model, encoders, features

model, encoders, features = load_artifacts()

# UI
st.set_page_config(page_title="Propensity Scorer", layout="wide")
st.title("Propensity-to-Buy Scorer")
st.markdown("Enter lead details → Get **real-time** conversion probability + **SHAP explanation**")

# Input form
with st.form("lead_form"):
    col1, col2 = st.columns(2)
    with col1:
        clv = st.number_input("Customer Lifetime Value ($)", min_value=0.0, value=8000.0, step=100.0)
        premium = st.number_input("Monthly Premium Auto ($)", min_value=0.0, value=100.0, step=10.0)
        months = st.number_input("Months Since Policy Inception", min_value=0, value=12, step=1)
    with col2:
        channel = st.selectbox("Sales Channel", options=encoders['Sales Channel'].classes_)
        offer = st.selectbox("Renew Offer Type", options=encoders['Renew Offer Type'].classes_)
        education = st.selectbox("Education", options=encoders['Education'].classes_)

    col3, col4 = st.columns(2)
    with col3:
        employment = st.selectbox("Employment Status", options=encoders['EmploymentStatus'].classes_)
        gender = st.selectbox("Gender", options=encoders['Gender'].classes_)
        location = st.selectbox("Location Code", options=encoders['Location Code'].classes_)
    with col4:
        coverage = st.selectbox("Coverage", options=["Basic", "Extended", "Premium"])
        vehicle = st.text_input("Vehicle Class", value="Two-Door Car")

    submitted = st.form_submit_button("Score Lead")

# Predict
if submitted:
    with st.spinner("Scoring..."):
        # Build feature vector
        data = {
            'Customer Lifetime Value': clv,
            'Monthly Premium Auto': premium,
            'Months_Since_Start': months,
            'Premium_Policy': 1 if coverage == 'Premium' else 0,
            'Luxury_Vehicle': 1 if 'Luxury' in vehicle else 0
        }
        for col in ['Sales Channel', 'Renew Offer Type', 'Education', 'EmploymentStatus', 'Gender', 'Location Code']:
            le = encoders[col]
            val = locals()[col.lower().replace(' ', '_')]
            data[col] = le.transform([val])[0] if val in le.classes_ else 0

        X = pd.DataFrame([data])[features]
        prob = model.predict_proba(X)[0, 1]

        # Results
        st.success(f"**Propensity Score: {prob:.1%}**")
        action = "Route to Agent" if prob > 0.7 else "Web Funnel"
        st.metric("Recommended Action", action)

        # SHAP Explanation
        with st.spinner("Generating SHAP explanation..."):
            try:
                # Use model-agnostic explainer
                predict_fn = lambda x: model.predict_proba(x)[:, 1]
                explainer = shap.Explainer(predict_fn, X, feature_names=features)
                shap_values = explainer(X)

                fig, ax = plt.subplots()
                shap.waterfall_plot(shap_values[0], max_display=10, show=False)
                st.pyplot(fig)
            except:
                st.warning("SHAP plot skipped (optional)")

Upload your .pkl files:


Saving feature_names.pkl to feature_names (1).pkl
Saving label_encoders.pkl to label_encoders (1).pkl
Saving propensity_model.pkl to propensity_model (1).pkl


UsageError: Line magic function `%%writefile` not found.


# PROJECT STRUCTURE (GitHub-Ready)

In [ ]:
df.columns

Index(['Customer', 'State', 'Customer Lifetime Value', 'Response', 'Coverage',
       'Education', 'Effective To Date', 'EmploymentStatus', 'Gender',
       'Income', 'Location Code', 'Marital Status', 'Monthly Premium Auto',
       'Months Since Last Claim', 'Months Since Policy Inception',
       'Number of Open Complaints', 'Number of Policies', 'Policy Type',
       'Policy', 'Renew Offer Type', 'Sales Channel', 'Total Claim Amount',
       'Vehicle Class', 'Vehicle Size'],
      dtype='object')

In [ ]:
df.shape

(9134, 24)

In [ ]:
df.isna().sum()

,0
Customer,0
State,0
Customer Lifetime Value,0
Response,0
Coverage,0
Education,0
Effective To Date,0
EmploymentStatus,0
Gender,0
Income,0


In [ ]:
# train.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten
import joblib
import os

# -------------------------------
# 1. Load & Prepare Data
# -------------------------------
print("Loading data...")
df = pd.read_csv("data.csv")

# Target: Response = Yes → Converted = 1
df["Converted"] = (df["Response"] == "Yes").astype(int)

# Drop unnecessary columns
drop_cols = ["Customer", "Effective To Date", "Response"]  # ID, date, original target
X = df.drop(columns=drop_cols + ["Converted"])
y = df["Converted"]

print(f"Dataset: {X.shape[0]} rows, {X.shape[1]} features")
print(f"Conversion Rate: {y.mean():.2%}")

Loading data...
Dataset: 9134 rows, 21 features
Conversion Rate: 14.32%


In [ ]:

# -------------------------------
# 2. Encode Categorical Features
# -------------------------------
encoders = {}
X_encoded = X.copy()

categorical_cols = X.select_dtypes(include="object").columns
for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

# Save encoders & feature names
joblib.dump(encoders, "label_encoders.pkl")
joblib.dump(X_encoded.columns.tolist(), "feature_names.pkl")

# -------------------------------
# 3. Train-Test Split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, stratify=y, random_state=42
)

# Save test set for comparison
X_test.to_csv("X_test.csv", index=False)
pd.DataFrame(y_test).to_csv("y_test.csv", index=False)

# -------------------------------
# 4. Model 1: XGBoost
# -------------------------------
print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="auc"
)
xgb_model.fit(X_train, y_train)
joblib.dump(xgb_model, "xgboost_model.pkl")

Training XGBoost...


['xgboost_model.pkl']

In [ ]:
!pip install scikeras

In [ ]:
!pip install ctgan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.3 MB/s eta 0:00:00


In [ ]:
!pip install ctgan==0.7.3

ERROR: Ignored the following versions that require a different python version: 0.2.2 Requires-Python >=3.6,<3.9; 0.2.2.dev1 Requires-Python >=3.5,<3.9; 0.2.2.dev2 Requires-Python >=3.6,<3.9; 0.2.2.dev3 Requires-Python >=3.6,<3.9; 0.3.0 Requires-Python >=3.6,<3.9; 0.3.0.dev0 Requires-Python >=3.5,<3.9; 0.3.0.dev1 Requires-Python >=3.6,<3.9; 0.3.1 Requires-Python >=3.6,<3.9; 0.3.1.dev0 Requires-Python >=3.6,<3.9; 0.3.1.dev1 Requires-Python >=3.6,<3.9; 0.3.1.dev2 Requires-Python >=3.6,<3.9; 0.3.2.dev0 Requires-Python >=3.6,<3.9; 0.4.0 Requires-Python >=3.6,<3.9; 0.4.0.dev0 Requires-Python >=3.6,<3.9; 0.4.0.dev1 Requires-Python >=3.6,<3.9; 0.4.1 Requires-Python >=3.6,<3.9; 0.4.1.dev0 Requires-Python >=3.6,<3.9; 0.4.1.dev1 Requires-Python >=3.6,<3.9; 0.4.2 Requires-Python >=3.6,<3.9; 0.4.2.dev0 Requires-Python >=3.6,<3.9; 0.4.3 Requires-Python >=3.6,<3.9; 0.4.3.dev0 Requires-Python >=3.6,<3.9; 0.4.3.dev1 Requires-Python >=3.6,<3.9; 0.4.4.dev0 Requires-Python >=3.6,<3.9; 0.5.0 Requires-Pytho

In [ ]:
!pip install ctgan==0.10.2 pandas numpy scikit-learn xgboost imbalanced-learn joblib tensorflow --quiet

In [ ]:
# Paste this entire block
%%writefile train.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten
import joblib
from ctgan import CTGANSynthesizer

print("Loading data...")
df = pd.read_csv("data.csv")
df["Converted"] = (df["Response"] == "Yes").astype(int)

drop_cols = ["Customer", "Effective To Date", "Response"]
X = df.drop(columns=drop_cols + ["Converted"])
y = df["Converted"]

print(f"Original: {X.shape[0]} rows, {y.mean():.2%} conversion")

# 1. SPLIT FIRST
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_test_raw.to_csv("X_test.csv", index=False)
pd.DataFrame(y_test).to_csv("y_test.csv", index=False)

# 2. ENCODE
encoders = {}
X_train = X_train_raw.copy()
X_test = X_test_raw.copy()

cat_cols = X_train.select_dtypes(include="object").columns
for col in cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train[col] = oe.fit_transform(X_train[[col]])
    X_test[col] = oe.transform(X_test[[col]])
    encoders[col] = oe

joblib.dump(encoders, "label_encoders.pkl")
joblib.dump(X_train.columns.tolist(), "feature_names.pkl")

# 3. GAN: Generate synthetic Converted=1
print("Training CTGAN...")
converted_train = X_train[y_train == 1].copy()
converted_train['Converted'] = 1

ctgan = CTGANSynthesizer(epochs=300, batch_size=64, verbose=False)
ctgan.fit(converted_train, discrete_columns=cat_cols.tolist() + ['Converted'])

n_synthetic = len(X_train[y_train == 0]) - len(X_train[y_train == 1])
synthetic = ctgan.sample(n_synthetic)
synthetic = synthetic.drop(columns=['Converted'])

print(f"Generated {n_synthetic} synthetic samples")

# 4. Combine
X_train_balanced = pd.concat([
    X_train[y_train == 1], synthetic, X_train[y_train == 0]
], ignore_index=True)
y_train_balanced = pd.Series([1] * (len(synthetic) + len(X_train[y_train == 1])) + [0] * len(X_train[y_train == 0]))

# 5. TRAIN
print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(n_estimators=300, max_depth=6, random_state=42)
xgb_model.fit(X_train_balanced, y_train_balanced)
joblib.dump(xgb_model, "xgboost_model.pkl")

print("Training RF...")
rf_model = RandomForestClassifier(n_estimators=400, max_depth=8, random_state=42)
rf_model.fit(X_train_balanced, y_train_balanced)
joblib.dump(rf_model, "rf_model.pkl")

print("Training CNN...")
X_train_cnn = X_train_balanced.values.reshape(-1, X_train_balanced.shape[1], 1)
cnn = Sequential([
    Conv1D(32, 3, activation='relu', input_shape=(X_train_balanced.shape[1], 1)),
    MaxPooling1D(2),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
cnn.fit(X_train_cnn, y_train_balanced, epochs=15, batch_size=32, verbose=0)
cnn.save("cnn_model.keras")

print("Training Stacking...")
base = [
    ('xgb', xgb.XGBClassifier(n_estimators=200, max_depth=5, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=7, random_state=42))
]
stack = StackingClassifier(estimators=base, final_estimator=LogisticRegression(max_iter=1000), cv=3)
stack.fit(X_train_balanced, y_train_balanced)
joblib.dump(stack, "stack_model.pkl")

print("All models saved!")

Writing train.py


In [ ]:
%%writefile compare.py
import joblib
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import tensorflow as tf

X_test = pd.read_csv("X_test.csv")
y_test = pd.read_csv("y_test.csv").values.ravel()

encoders = joblib.load("label_encoders.pkl")
for col, oe in encoders.items():
    X_test[col] = oe.transform(X_test[[col]])

models = {
    "XGBoost (GAN)": joblib.load("xgboost_model.pkl"),
    "Random Forest": joblib.load("rf_model.pkl"),
    "CNN": tf.keras.models.load_model("cnn_model.keras"),
    "Stacking": joblib.load("stack_model.pkl")
}

results = []
for name, model in models.items():
    print(f"Predicting {name}...")
    if "CNN" in name:
        X_cnn = X_test.values.reshape(-1, X_test.shape[1], 1)
        prob = model.predict(X_cnn, verbose=0).ravel()
    else:
        prob = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, prob)
    ap = average_precision_score(y_test, prob)
    lift = np.mean(y_test[prob >= np.percentile(prob, 90)]) / y_test.mean()

    results.append({"Model": name, "AUC": round(auc, 4), "PR-AUC": round(ap, 4), "Lift@10%": round(lift, 2)})

df = pd.DataFrame(results).sort_values("AUC", ascending=False)
print("\n" + "="*60)
print("GAN-BALANCED MODEL COMPARISON")
print("="*60)
print(df.to_string(index=False))

best = df.iloc[0]["Model"]
best_model = models[best]
if "CNN" in best:
    best_model.save("best_model.keras")
else:
    joblib.dump(best_model, "best_model.pkl")
print(f"\nWINNER: {best}")

Writing compare.py


In [ ]:
!python train.py

2025-11-02 07:10:01.035769: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762067401.103613  226161 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762067401.124701  226161 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762067401.200692  226161 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762067401.200816  226161 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762067401.200828  226161 computation_placer.cc:177] computation placer alr

In [ ]:
!python compare.py

2025-11-02 07:10:53.463735: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762067453.489821  226392 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762067453.497626  226392 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762067453.517838  226392 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762067453.517900  226392 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762067453.517904  226392 computation_placer.cc:177] computation placer alr

In [ ]:
from google.colab import files
for f in ["best_model.pkl", "label_encoders.pkl", "feature_names.pkl",
          "xgboost_model.pkl", "rf_model.pkl", "cnn_model.keras", "stack_model.pkl"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>